In [ ]:
import gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

In [ ]:
import sys
sys.path.append("../")

In [ ]:
#from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
#from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
#from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
#from mbt_gym.gym.TradingEnvironment import TradingEnvironment
#from mbt_gym.gym.wrappers import *
#from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
#from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel, GeometricBrownianMotionMidpriceModel
#from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
#from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel, SynchronousHawkesArrivalModel # my addition 
#from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
#from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction, DynamicLOBExponentialFillFunction# my addition 
#from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
#from mbt_gym.gym.Bek_TradingEnvironment import Bek_TradingEnvironment # my addition
#from mbt_gym.gym.Bek_ModelDynamics import Bek_LimitOrderModelDynamics # my addition
#from mbt_gym.stochastic_processes.arrival_models import SynchronousLOBDepthModel #


 MULTI-ASSET (N=2) Case

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 

# MULTI-ASSET (N=2) using the MULTI-ASSET codebase checkpoints
# Contour plots (side-by-side) + separate colorbars
# SAME colormap + SAME vmin/vmax + save data + save metadata (TXT + JSON)
# Works for BOTH: checkpoint_path OR best_model_path
# If best_model_path is loaded -> run_tag becomes: <trial_folder>_Best_Model
# Produces ONE figure per asset: [delta_minus | delta_plus]
# The inventory of the other asset is fixed at zero
# ============================

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from matplotlib.ticker import FormatStrFormatter  # <-- ADDED

# ----------------------------
# GLOBAL STYLE (LaTeX-like CMR)
# ----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "axes.titlesize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
})

# ----------------------------
# Paths (edit these)
# ----------------------------
#checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial20/PPO_2Assets_Trial20_100M.zip")
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial1/PPO_2Assets_Trial1_100M.zip")
best_model_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Best_2Assets/PPO_2Assets_Trial1/best_model.zip")

# ----------------------------
# Choose what you load (set ONE)
# ----------------------------
use_best_model = False

model_path = best_model_path if use_best_model else checkpoint_path
model = PPO.load(model_path)

# ----------------------------
# Infer run_tag
# ----------------------------
def infer_run_tag_for_best(best_path: str) -> str:
    trial_folder = os.path.basename(os.path.dirname(best_path))
    return f"{trial_folder}_Best_Model"

def infer_run_tag_for_checkpoint(ckpt_path: str) -> str:
    return os.path.splitext(os.path.basename(ckpt_path))[0]

run_tag = infer_run_tag_for_best(best_model_path) if use_best_model else infer_run_tag_for_checkpoint(checkpoint_path)

# ----------------------------
# Infer trial folder
# ----------------------------
def infer_trial_folder_from_model_path(model_path: str, use_best: bool) -> str:
    return os.path.basename(os.path.dirname(model_path))

trial_folder = infer_trial_folder_from_model_path(model_path, use_best_model)

# ----------------------------
# Save dirs
# ----------------------------
assets_tag = trial_folder.split("_")[1]

fig_root  = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/contour")
data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/contour_data")

fig_dir  = os.path.join(fig_root, trial_folder)
data_dir = os.path.join(data_root, trial_folder)

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print("Saving under:")
print("  fig_dir :", fig_dir)
print("  data_dir:", data_dir)

# ----------------------------
# Fixed values + grid
# ----------------------------
num_assets = 2

fixed_state_values = np.array([
    15.25, 15.25,
    17.49, 17.49,
    3.55,  3.55,
    4.12,  4.12
], dtype=float)

q_values = np.arange(-10, 11, 1)
t_values = np.linspace(0, 1, 101)

obs_dim = None
if getattr(model, "observation_space", None) is not None:
    obs_dim = model.observation_space.shape[0]

print("Detected obs_dim:", obs_dim)

# ----------------------------
# Helper
# ----------------------------
def build_obs_for_asset(asset_index: int, q: float, t: float) -> np.ndarray:
    if asset_index == 0:
        q_vec = np.array([q, 0.0], dtype=float)
    else:
        q_vec = np.array([0.0, q], dtype=float)

    if obs_dim == 3:
        obs = np.concatenate([q_vec, [t]]).reshape(1, -1)
    elif obs_dim == 11:
        obs = np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)
    else:
        obs = np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)

    return obs

# ----------------------------
# Plot settings
# ----------------------------
cmap_shared = "viridis"

# ----------------------------
# Main loop
# ----------------------------
for asset in range(num_assets):

    delta_minus = np.zeros((len(q_values), len(t_values)))
    delta_plus  = np.zeros((len(q_values), len(t_values)))

    for i, q in enumerate(q_values):
        for j, t in enumerate(t_values):
            obs = build_obs_for_asset(asset, q, t)
            action, _ = model.predict(obs, deterministic=True)
            action = np.asarray(action).reshape(1, num_assets, 2)

            delta_minus[i, j] = action[0, asset, 0]
            delta_plus[i, j]  = action[0, asset, 1]

    print(f"\nAsset {asset+1}:")
    print("  delta_minus min/max:", float(delta_minus.min()), float(delta_minus.max()))
    print("  delta_plus  min/max:", float(delta_plus.min()),  float(delta_plus.max()))

    vmin = float(min(delta_minus.min(), delta_plus.min()))
    vmax = float(max(delta_minus.max(), delta_plus.max()))
    levels = np.linspace(vmin, vmax, 60)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    label_fs = 26
    title_fs = 26
    cbar_fs = 16

    # δ-
    cf1 = axes[0].contourf(
        t_values, q_values, delta_minus,
        levels=levels, cmap=cmap_shared, vmin=vmin, vmax=vmax
    )

    axes[0].set_title(rf"$\delta_{{{asset+1}}}^{{-}}$", fontsize=title_fs)
    axes[0].set_xlabel("Time (t)", fontsize=label_fs)
    axes[0].set_ylabel(rf"Inventory ($q_{{{asset+1}}}$)", fontsize=label_fs)

    axes[0].tick_params(axis='both', which='major', width=1.4, length=6)
    for label in axes[0].get_xticklabels() + axes[0].get_yticklabels():
        label.set_fontweight('medium')

    cbar1 = fig.colorbar(cf1, ax=axes[0], pad=0.02)
    cbar1.ax.tick_params(labelsize=cbar_fs, width=1.2)

    # ----------- NEW (ticks + formatting) -----------
    num_ticks = 5
    ticks = np.linspace(vmin, vmax, num_ticks)
    cbar1.set_ticks(ticks)
    cbar1.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    # -----------------------------------------------

    # δ+
    cf2 = axes[1].contourf(
        t_values, q_values, delta_plus,
        levels=levels, cmap=cmap_shared, vmin=vmin, vmax=vmax
    )

    axes[1].set_title(rf"$\delta_{{{asset+1}}}^{{+}}$", fontsize=title_fs)
    axes[1].set_xlabel("Time (t)", fontsize=label_fs)
    axes[1].set_ylabel(rf"Inventory ($q_{{{asset+1}}}$)", fontsize=label_fs)

    axes[1].tick_params(axis='both', which='major', width=1.4, length=6)
    for label in axes[1].get_xticklabels() + axes[1].get_yticklabels():
        label.set_fontweight('medium')

    cbar2 = fig.colorbar(cf2, ax=axes[1], pad=0.02)
    cbar2.ax.tick_params(labelsize=cbar_fs, width=1.2)

    # ----------- NEW (ticks + formatting) -----------
    cbar2.set_ticks(ticks)
    cbar2.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    # -----------------------------------------------

    # ----------------------------
    # SAVE
    # ----------------------------
    asset_tag = f"Asset{asset+1}"

    fig_path_pdf = os.path.join(fig_dir,  f"{run_tag}_{asset_tag}.pdf")
    fig_path_png = os.path.join(fig_dir,  f"{run_tag}_{asset_tag}.png")
    fig.savefig(fig_path_pdf, bbox_inches="tight")
    fig.savefig(fig_path_png, dpi=600, bbox_inches="tight")

    dm_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_minus.txt")
    dp_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_plus.txt")
    np.savetxt(dm_path, delta_minus, delimiter=",")
    np.savetxt(dp_path, delta_plus, delimiter=",")

    meta_txt_path  = os.path.join(data_dir, f"{run_tag}_{asset_tag}_meta.txt")
    meta_json_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_meta.json")

    meta = {
        "run_tag": run_tag,
        "trial_folder": trial_folder,
        "asset_index": int(asset),
        "asset_tag": asset_tag,
        "use_best_model": bool(use_best_model),
        "source_model_path": model_path,
        "obs_dim_detected": int(obs_dim) if obs_dim is not None else None,
    }

    with open(meta_json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    with open(meta_txt_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(meta, indent=2))

    print("\nSaved:")
    print("  Figure PDF:", fig_path_pdf)
    print("  Figure PNG:", fig_path_png)

    plt.show()

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 

# OVER TIME (δ⁻ and δ⁺) : Kinda horizontal line slices of the contor plots
# ONE FIGURE PER ASSET:
# The inventory of the other asset is fixed at zero
# Standalone: loads saved arrays + rebuilds q_values/t_values + smooth lines
# NOW WITH: per-trial subfolder inferred from run_tag (e.g., PPO_2Assets_Trial3)
# Expects:
#   data_dir = ...\MultiAsset\contour_data\<trial_folder>\
# Saves to:
#   out_dir  = ...\MultiAsset\over_time\<trial_folder>\
# ============================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# GLOBAL STYLE (LaTeX-like CMR)
# ----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26, #18
    "xtick.labelsize": 20, #14
    "ytick.labelsize": 20, #14
    "axes.linewidth": 1.2, # 1.2
    "lines.linewidth": 6, # 3.5 
})

# ----------------------------
# PATHS (edit these)
# ----------------------------
data_root = os.path.join(PROJECT_ROOT, "N_figures/2Assets/contour_data")
out_root  = os.path.join(PROJECT_ROOT, "N_figures/2Assets/over_time")

#run_tag = "PPO_2Assets_Trial1_100M"  # <-- must match files saved by your contour script
run_tag = "PPO_2Assets_Trial1_100M"  # <-- must match files saved by your contour script
num_assets = 2

# ----------------------------
# Infer trial folder from run_tag
# ----------------------------
def infer_trial_folder_from_run_tag(tag: str) -> str:
    m = re.match(r"^(PPO_\d+Assets_Trial\d+)", tag)
    if m:
        return m.group(1)
    parts = tag.split("_")
    return "_".join(parts[:-1]) if len(parts) >= 2 else tag

trial_folder = infer_trial_folder_from_run_tag(run_tag)

# Final dirs
data_dir = os.path.join(data_root, trial_folder)
out_dir  = os.path.join(out_root, trial_folder)
os.makedirs(out_dir, exist_ok=True)

print("Using:")
print("  trial_folder:", trial_folder)
print("  data_dir     :", data_dir)
print("  out_dir      :", out_dir)

# ----------------------------
# GRID (must match contour script)
# ----------------------------
q_values = np.arange(-10, 11, 1)
t_values = np.linspace(0, 1, 101)

# ----------------------------
# Choose q slices to visualize
# ----------------------------
q_subset = np.arange(-3, 4, 1)

# ----------------------------
# Line styles for different q
# ----------------------------
line_styles = ["-", "--", "-.", ":", (0, (3,1,1,1)), (0, (5,1)), (0, (1,1))]

# ----------------------------
# (Optional) smoothing helper
# ----------------------------
def smooth_1d(y: np.ndarray, window: int = 7) -> np.ndarray:
    if window is None or window <= 1:
        return y
    window = int(window)
    if window % 2 == 0:
        window += 1
    if window >= len(y):
        window = max(3, len(y) // 2 * 2 + 1)

    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(ypad, kernel, mode="valid")

smooth_window = 1

# ----------------------------
# Loop over assets
# ----------------------------
for asset in range(num_assets):
    asset_tag = f"Asset{asset+1}"

    dm_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_minus.txt")
    dp_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_plus.txt")

    if not os.path.exists(dm_path):
        raise FileNotFoundError(f"Missing file: {dm_path}")
    if not os.path.exists(dp_path):
        raise FileNotFoundError(f"Missing file: {dp_path}")

    delta_minus = np.loadtxt(dm_path, delimiter=",")
    delta_plus  = np.loadtxt(dp_path, delimiter=",")

    expected_shape = (len(q_values), len(t_values))
    if delta_minus.shape != expected_shape:
        raise ValueError(f"{asset_tag} delta_minus shape mismatch.")
    if delta_plus.shape != expected_shape:
        raise ValueError(f"{asset_tag} delta_plus shape mismatch.")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    # δ⁻ panel
    for i, q in enumerate(q_subset):
        idx = np.where(q_values == q)[0]
        if len(idx) == 0:
            continue
        q_index = idx[0]
        y = delta_minus[q_index, :]
        y_s = smooth_1d(y, window=smooth_window)

        linestyle = line_styles[i % len(line_styles)]
        label_name = rf"$q_{1 if asset == 0 else 2}={q}$"

        axes[0].plot(t_values, y_s, linestyle=linestyle, label=label_name)

    #axes[0].set_title(...)
    axes[0].set_xlabel("Time (t)")
    axes[0].set_ylabel(rf"$\delta_{{{asset+1}}}^-$")
    axes[0].grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
    axes[0].tick_params(width=1.5, length=6)

    for label in axes[0].get_xticklabels() + axes[0].get_yticklabels():
        label.set_fontweight('medium')

    axes[0].legend(fontsize=16, ncol=2, frameon=True)

    # δ⁺ panel
    for i, q in enumerate(q_subset):
        idx = np.where(q_values == q)[0]
        if len(idx) == 0:
            continue
        q_index = idx[0]
        y = delta_plus[q_index, :]
        y_s = smooth_1d(y, window=smooth_window)

        linestyle = line_styles[i % len(line_styles)]
        label_name = rf"$q_{1 if asset == 0 else 2}={q}$"

        axes[1].plot(t_values, y_s, linestyle=linestyle, label=label_name)

    #axes[1].set_title(...)
    axes[1].set_xlabel("Time (t)")
    axes[1].set_ylabel(rf"$\delta_{{{asset+1}}}^+$")
    axes[1].grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
    axes[1].tick_params(width=1.5, length=6)

    for label in axes[1].get_xticklabels() + axes[1].get_yticklabels():
        label.set_fontweight('medium')

    axes[1].legend(fontsize=16, ncol=2, frameon=True)

    #fig.suptitle(...)

    ymin = float(min(delta_minus.min(), delta_plus.min()))
    ymax = float(max(delta_minus.max(), delta_plus.max()))
    axes[0].set_ylim(ymin, ymax)
    axes[1].set_ylim(ymin, ymax)

    pdf_path = os.path.join(out_dir, f"{run_tag}_{asset_tag}_over_time_smooth.pdf")
    png_path = os.path.join(out_dir, f"{run_tag}_{asset_tag}_over_time_smooth.png")

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"\nSaved over-time plots for {asset_tag}:")
    print("  PDF:", pdf_path)
    print("  PNG:", png_path)

    plt.show()

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# MID TIME 
# STANDALONE: PLOT δ⁻ AND δ⁺ OVER q AT t = 0.5 (N = 2 assets)
# # The inventory of the other asset is fixed at zero
# - loads saved per-asset delta arrays from contour_data\<trial_folder>\
# - rebuilds q_values/t_values grid
# - saves PDF + high-res PNG under ...\MultiAsset\mid_time\<trial_folder>\
# ============================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# GLOBAL STYLE (LaTeX-like CMR)
# ----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ----------------------------
# PATHS (edit these)
# ----------------------------
data_root = os.path.join(PROJECT_ROOT, "N_figures/2Assets/contour_data")
out_root  = os.path.join(PROJECT_ROOT, "N_figures/2Assets/mid_time")

run_tag = "PPO_2Assets_Trial1_100M"  # <-- must match saved files in contour_data
num_assets = 2

# ----------------------------
# Infer trial folder from run_tag
# ----------------------------
def infer_trial_folder_from_run_tag(tag: str) -> str:
    m = re.match(r"^(PPO_\d+Assets_Trial\d+)", tag)
    if m:
        return m.group(1)
    parts = tag.split("_")
    return "_".join(parts[:-1]) if len(parts) >= 2 else tag

trial_folder = infer_trial_folder_from_run_tag(run_tag)

# Final dirs
data_dir = os.path.join(data_root, trial_folder)
out_dir  = os.path.join(out_root, trial_folder)
os.makedirs(out_dir, exist_ok=True)

print("Using:")
print("  trial_folder:", trial_folder)
print("  data_dir     :", data_dir)
print("  out_dir      :", out_dir)

# ----------------------------
# GRID (must match contour script)
# ----------------------------
q_values = np.arange(-10, 11, 1)
t_values = np.linspace(0, 1, 101)

# ----------------------------
# Pick t = 0.5 (closest grid point)
# ----------------------------
t_target = 0.5
t_index = int(np.argmin(np.abs(t_values - t_target)))
t_used = float(t_values[t_index])
print(f"Using t_index = {t_index}, t_value = {t_used}")

# ----------------------------
# Loop through assets
# ----------------------------
for asset in range(num_assets):
    asset_tag = f"Asset{asset+1}"

    # ----------------------------
    # Load saved arrays for this asset (from trial subfolder)
    # ----------------------------
    dm_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_minus.txt")
    dp_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_plus.txt")

    if not os.path.exists(dm_path):
        raise FileNotFoundError(f"Missing file: {dm_path}")
    if not os.path.exists(dp_path):
        raise FileNotFoundError(f"Missing file: {dp_path}")

    delta_minus = np.loadtxt(dm_path, delimiter=",")
    delta_plus  = np.loadtxt(dp_path, delimiter=",")

    # ----------------------------
    # Sanity checks
    # ----------------------------
    expected_shape = (len(q_values), len(t_values))
    if delta_minus.shape != expected_shape:
        raise ValueError(
            f"{asset_tag} delta_minus shape {delta_minus.shape} != expected {expected_shape}. "
            f"Check q_values/t_values here vs how contour data was saved."
        )
    if delta_plus.shape != expected_shape:
        raise ValueError(
            f"{asset_tag} delta_plus shape {delta_plus.shape} != expected {expected_shape}. "
            f"Check q_values/t_values here vs how contour data was saved."
        )

    # ----------------------------
    # Extract curves at mid-time
    # ----------------------------
    delta_minus_t = delta_minus[:, t_index]
    delta_plus_t  = delta_plus[:,  t_index]

    # ----------------------------
    # Plot (smooth line style, no markers)
    # ----------------------------
    plt.figure(figsize=(12, 6))

    if asset == 0:
        plt.plot(q_values, delta_minus_t, linestyle="-",  label=r"$\delta_1^-$")
        plt.plot(q_values, delta_plus_t,  linestyle="--", label=r"$\delta_1^+$")
        #ylabel = r"Quote Distance ($\delta_1^-, \delta_1^+$)"
        ylabel = r"$\delta_1^-, \delta_1^+$"
        xlabel = r"Inventory ($q_1$)"
    else:
        plt.plot(q_values, delta_minus_t, linestyle="-",  label=r"$\delta_2^-$")
        plt.plot(q_values, delta_plus_t,  linestyle="--", label=r"$\delta_2^+$")
        #ylabel = r"Quote Distance ($\delta_2^-, \delta_2^+$)"
        ylabel = r"$\delta_2^-, \delta_2^+$"
        xlabel = r"Inventory ($q_2$)"

    #plt.title(...)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
    plt.tick_params(width=1.5, length=6)

    for label in plt.gca().get_xticklabels() + plt.gca().get_yticklabels():
        label.set_fontweight('medium')

    plt.legend(fontsize=25)
    plt.tight_layout()

    # ----------------------------
    # Save (publication quality) into trial subfolder
    # ----------------------------
    t_str = f"{int(round(t_used * 100)):03d}"  # <-- NEW: 0.5 -> "050"

    pdf_path = os.path.join(out_dir, f"{run_tag}_{asset_tag}_mid_time_t{t_str}.pdf")
    png_path = os.path.join(out_dir, f"{run_tag}_{asset_tag}_mid_time_t{t_str}.png")

    plt.savefig(pdf_path, bbox_inches="tight")
    plt.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"\nSaved mid-time plot for {asset_tag}:")
    print("  PDF:", pdf_path)
    print("  PNG:", png_path)

    plt.show()

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# NOTE: not used in the (first version of the Manuscript submitted to Fututes Market) 
# 2D INVENTORY HEATMAPS at fixed time t = t_target
# For N=2 assets:
#   Heatmaps over (q1, q2) for each asset's quotes:
#     Asset 1: δ1^- (q1,q2), δ1^+ (q1,q2)
#     Asset 2: δ2^- (q1,q2), δ2^+ (q1,q2)
#
# Produces ONE figure per asset (side-by-side δ^- | δ^+), with separate colorbars.
# Saves:
#   ...\N_figures\<kAssets>\inv_heatmaps\<trial_folder>\
#   ...\N_figures\<kAssets>\inv_heatmaps_data\<trial_folder>\
#
# Works for BOTH checkpoint_path OR best_model_path
# ============================================

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

# ----------------------------
# Paths (edit these)
# ----------------------------
#checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial1/PPO_2Assets_Trial1_100M.zip")
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial20/PPO_2Assets_Trial20_100M.zip")
best_model_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Best_2Assets/PPO_2Assets_Trial1/best_model.zip")

# ----------------------------
# Choose what you load (set ONE)
# ----------------------------
use_best_model = False  # True => load best_model.zip, False => load checkpoint

model_path = best_model_path if use_best_model else checkpoint_path
model = PPO.load(model_path)

# ----------------------------
# Infer run_tag
# ----------------------------
def infer_run_tag_for_best(best_path: str) -> str:
    trial_folder = os.path.basename(os.path.dirname(best_path))  # PPO_2Assets_TrialX
    return f"{trial_folder}_Best_Model"

def infer_run_tag_for_checkpoint(ckpt_path: str) -> str:
    return os.path.splitext(os.path.basename(ckpt_path))[0]

run_tag = infer_run_tag_for_best(best_model_path) if use_best_model else infer_run_tag_for_checkpoint(checkpoint_path)

# ----------------------------
# Trial folder (same as your contour code)
# ----------------------------
def infer_trial_folder_from_model_path(model_path: str) -> str:
    return os.path.basename(os.path.dirname(model_path))

trial_folder = infer_trial_folder_from_model_path(model_path)

# ----------------------------
# Assets tag (e.g., "2Assets") from trial_folder: "PPO_2Assets_Trial1"
# ----------------------------
assets_tag = trial_folder.split("_")[1]  # "2Assets"

# ----------------------------
# Save dirs (auto-create)
# ----------------------------
fig_root  = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/inv_heatmaps")
data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/inv_heatmaps_data")

fig_dir  = os.path.join(fig_root, trial_folder)
data_dir = os.path.join(data_root, trial_folder)

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print("Saving under:")
print("  fig_dir :", fig_dir)
print("  data_dir:", data_dir)

# ----------------------------
# Fixed values (same convention as your contour script)
# Reduced obs (extended) for N=2:
# obs = [q1, q2, t, λ1_bid, λ1_ask, λ2_bid, λ2_ask, c1_bid, c1_ask, c2_bid, c2_ask]
# ----------------------------
num_assets = 2

#fixed_state_values = np.array([
    #15.25, 15.25,
    #17.49, 17.49,
   # 3.55,  3.55,
   # 4.12,  4.12
#], dtype=float)


fixed_state_values = np.array([
    21.63, 21.63,
    22.44, 22.44,
    4.00,  4.00,
    4.44,  4.44
], dtype=float)

# ----------------------------
# Choose grid over inventories
# ----------------------------
q_values = np.arange(-10, 11, 1)  # q1 and q2 grid: -10..10

# Choose a fixed time slice
t_target = 0.5  # you can change to 0.25, 0.75, etc.

# Detect obs dim
obs_dim = None
if getattr(model, "observation_space", None) is not None:
    obs_dim = model.observation_space.shape[0]
print("Detected obs_dim:", obs_dim)

# ----------------------------
# Helper: build obs for N=2 WITHOUT freezing inventories
# ----------------------------
def build_obs(q1: float, q2: float, t: float) -> np.ndarray:
    q_vec = np.array([q1, q2], dtype=float)

    if obs_dim == 3:
        # [q1,q2,t]
        return np.concatenate([q_vec, [t]]).reshape(1, -1)

    elif obs_dim == 11:
        # [q1,q2,t, fixed_state_values(8)]
        return np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)

    else:
        # fallback: assume extended
        return np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)

# ----------------------------
# Plot settings:
# - SAME colormap
# - SAME vmin/vmax across δ^- and δ^+ within each asset figure
# ----------------------------
cmap_shared = "viridis"

# ----------------------------
# Evaluate and plot for each asset
# ----------------------------
for asset in range(num_assets):
    asset_tag = f"Asset{asset+1}"

    # Arrays over (q1, q2)
    # shape: (len(q1), len(q2)) => (21,21)
    delta_minus = np.zeros((len(q_values), len(q_values)))
    delta_plus  = np.zeros((len(q_values), len(q_values)))

    for i, q1 in enumerate(q_values):
        for j, q2 in enumerate(q_values):
            obs = build_obs(q1, q2, t_target)
            action, _ = model.predict(obs, deterministic=True)
            action = np.asarray(action).reshape(1, num_assets, 2)

            delta_minus[i, j] = action[0, asset, 0]
            delta_plus[i, j]  = action[0, asset, 1]

    # diagnostics
    print(f"\n{asset_tag} @ t={t_target}:")
    print("  delta_minus min/max:", float(delta_minus.min()), float(delta_minus.max()))
    print("  delta_plus  min/max:", float(delta_plus.min()),  float(delta_plus.max()))

    # shared scaling across δ⁻ and δ⁺ for THIS asset
    vmin = float(min(delta_minus.min(), delta_plus.min()))
    vmax = float(max(delta_minus.max(), delta_plus.max()))

    # ----------------------------
    # Plot (imshow heatmaps) side-by-side
    # NOTE: imshow expects axes mapping:
    #   x-axis => q2 (columns)
    #   y-axis => q1 (rows)
    # We set extent accordingly.
    # ----------------------------
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    extent = [q_values.min(), q_values.max(), q_values.min(), q_values.max()]  # [x_min, x_max, y_min, y_max]

    im1 = axes[0].imshow(
        delta_minus,
        origin="lower",
        extent=extent,
        aspect="auto",
        cmap=cmap_shared,
        vmin=vmin,
        vmax=vmax,
    )
    axes[0].set_title(rf"{asset_tag}: $\delta^{{-}}$ over $(q_1, q_2)$ at $t={t_target}$", fontsize=12)
    axes[0].set_xlabel(r"Inventory $q_2$", fontsize=11)
    axes[0].set_ylabel(r"Inventory $q_1$", fontsize=11)
    axes[0].tick_params(labelsize=10)
    cbar1 = fig.colorbar(im1, ax=axes[0], pad=0.02)
    cbar1.set_label(r"$\delta^{-}$", fontsize=11)
    cbar1.ax.tick_params(labelsize=10)

    im2 = axes[1].imshow(
        delta_plus,
        origin="lower",
        extent=extent,
        aspect="auto",
        cmap=cmap_shared,
        vmin=vmin,
        vmax=vmax,
    )
    axes[1].set_title(rf"{asset_tag}: $\delta^{{+}}$ over $(q_1, q_2)$ at $t={t_target}$", fontsize=12)
    axes[1].set_xlabel(r"Inventory $q_2$", fontsize=11)
    axes[1].set_ylabel(r"Inventory $q_1$", fontsize=11)
    axes[1].tick_params(labelsize=10)
    cbar2 = fig.colorbar(im2, ax=axes[1], pad=0.02)
    cbar2.set_label(r"$\delta^{+}$", fontsize=11)
    cbar2.ax.tick_params(labelsize=10)

    fig.suptitle(f"Inventory heatmaps — {run_tag} — {asset_tag} — t={t_target}", fontsize=13)

    # ----------------------------
    # SAVE
    # ----------------------------
    # Figures
    fig_path_pdf = os.path.join(fig_dir, f"{run_tag}_{asset_tag}_inv_heatmap_t{t_target:.2f}.pdf")
    fig_path_png = os.path.join(fig_dir, f"{run_tag}_{asset_tag}_inv_heatmap_t{t_target:.2f}.png")
    fig.savefig(fig_path_pdf, bbox_inches="tight")
    fig.savefig(fig_path_png, dpi=600, bbox_inches="tight")

    # Data
    dm_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_minus_inv_heatmap_t{t_target:.2f}.txt")
    dp_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_plus_inv_heatmap_t{t_target:.2f}.txt")
    np.savetxt(dm_path, delta_minus, delimiter=",")
    np.savetxt(dp_path, delta_plus, delimiter=",")

    # Metadata
    meta_txt_path  = os.path.join(data_dir, f"{run_tag}_{asset_tag}_inv_heatmap_t{t_target:.2f}_meta.txt")
    meta_json_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_inv_heatmap_t{t_target:.2f}_meta.json")

    meta = {
        "run_tag": run_tag,
        "trial_folder": trial_folder,
        "assets_tag": assets_tag,
        "asset_index": int(asset),
        "asset_tag": asset_tag,
        "use_best_model": bool(use_best_model),
        "source_model_path": model_path,
        "obs_dim_detected": int(obs_dim) if obs_dim is not None else None,
        "t_target": float(t_target),

        "files": {
            "delta_minus_file": os.path.basename(dm_path),
            "delta_plus_file": os.path.basename(dp_path),
            "figure_pdf": os.path.basename(fig_path_pdf),
            "figure_png": os.path.basename(fig_path_png),
        },

        "grid": {
            "q_min": int(q_values.min()),
            "q_max": int(q_values.max()),
            "q_step": int(q_values[1] - q_values[0]),
            "q_num_points": int(len(q_values)),
            "q_axis_note": "Heatmaps use y-axis=q1, x-axis=q2.",
        },

        "fixed_state_values": {
            "note": (
                "If obs_dim == 3, only [q1,q2,t] was used. "
                "If obs_dim == 11, obs=[q1,q2,t,lambda1_bid,lambda1_ask,lambda2_bid,lambda2_ask,"
                "c1_bid,c1_ask,c2_bid,c2_ask]."
            ),
            "lambda1_bid": float(fixed_state_values[0]),
            "lambda1_ask": float(fixed_state_values[1]),
            "lambda2_bid": float(fixed_state_values[2]),
            "lambda2_ask": float(fixed_state_values[3]),
            "c1_bid": float(fixed_state_values[4]),
            "c1_ask": float(fixed_state_values[5]),
            "c2_bid": float(fixed_state_values[6]),
            "c2_ask": float(fixed_state_values[7]),
        },

        "plot_settings": {
            "colormap": str(cmap_shared),
            "vmin": float(vmin),
            "vmax": float(vmax),
        },
    }

    with open(meta_json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    with open(meta_txt_path, "w", encoding="utf-8") as f:
        for k in ["run_tag", "trial_folder", "assets_tag", "asset_tag", "use_best_model", "source_model_path", "obs_dim_detected", "t_target"]:
            f.write(f"{k}: {meta[k]}\n")
        f.write("\nFILES\n")
        for k, v in meta["files"].items():
            f.write(f"{k}: {v}\n")
        f.write("\nGRID\n")
        for k, v in meta["grid"].items():
            f.write(f"{k}: {v}\n")
        f.write("\nFIXED STATE VALUES\n")
        f.write(meta["fixed_state_values"]["note"] + "\n")
        for k in ["lambda1_bid","lambda1_ask","lambda2_bid","lambda2_ask","c1_bid","c1_ask","c2_bid","c2_ask"]:
            f.write(f"{k}: {meta['fixed_state_values'][k]}\n")
        f.write("\nPLOT SETTINGS\n")
        for k, v in meta["plot_settings"].items():
            f.write(f"{k}: {v}\n")

    print("\nSaved inventory heatmaps:")
    print("  Figure PDF:", fig_path_pdf)
    print("  Figure PNG:", fig_path_png)
    print("  Data delta_minus:", dm_path)
    print("  Data delta_plus :", dp_path)
    print("  Meta TXT:", meta_txt_path)
    print("  Meta JSON:", meta_json_path)

    plt.show()

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 

# N=2 case 
# Baiscally MIDTIME FIGURE but "other inventory" is not fixed at zero 
# ONE COMBINED FIGURE (1x2)
# δ_i^- and δ_i^+ vs q_i
# FULLY STANDALONE VERSION + METADATA
# ============================================

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

# ----------------------------
# GLOBAL STYLE (LaTeX-like CMR)
# ----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "axes.titlesize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
    "legend.fontsize": 16,
})

# ============================================================
# 1. PATHS (EDIT ONLY THIS SECTION)
# ============================================================
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial1/PPO_2Assets_Trial1_100M.zip")
best_model_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Best_2Assets/PPO_2Assets_Trial1/best_model.zip")

use_best_model = False
model_path = best_model_path if use_best_model else checkpoint_path

# ============================================================
# 2. LOAD MODEL
# ============================================================
model = PPO.load(model_path)

# ============================================================
# 3. RUN TAG + DIRECTORY LOGIC
# ============================================================
def infer_run_tag_for_best(best_path: str) -> str:
    trial_folder = os.path.basename(os.path.dirname(best_path))
    return f"{trial_folder}_Best_Model"

def infer_run_tag_for_checkpoint(ckpt_path: str) -> str:
    return os.path.splitext(os.path.basename(ckpt_path))[0]

def infer_trial_folder_from_model_path(model_path: str) -> str:
    return os.path.basename(os.path.dirname(model_path))

run_tag = infer_run_tag_for_best(best_model_path) if use_best_model else infer_run_tag_for_checkpoint(checkpoint_path)
trial_folder = infer_trial_folder_from_model_path(model_path)
assets_tag = trial_folder.split("_")[1]

# ============================================================
# 4. SAVE DIRECTORY
# ============================================================
fig_root  = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/line_slices")
data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/line_slices_data") # From Code 1

fig_dir   = os.path.join(fig_root, trial_folder)
data_dir  = os.path.join(data_root, trial_folder) # From Code 1

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True) # From Code 1

# ============================================================
# 5. OBSERVATION BUILDER
# ============================================================
num_assets = 2

obs_dim = None
if getattr(model, "observation_space", None) is not None:
    obs_dim = model.observation_space.shape[0]

print("Detected obs_dim:", obs_dim)

fixed_state_values = np.array([
    15.25, 15.25,
    17.49, 17.49,
    3.55,  3.55,
    4.12,  4.12
], dtype=float)

def build_obs(q1: float, q2: float, t: float) -> np.ndarray:
    q_vec = np.array([q1, q2], dtype=float)

    if obs_dim == 3:
        return np.concatenate([q_vec, [t]]).reshape(1, -1)
    elif obs_dim == 11:
        return np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)
    else:
        return np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)

# ============================================================
# 6. SETTINGS
# ============================================================
q_values = np.arange(-10, 11, 1)
fixed_other_values = [-10, -5, 0, 5, 10]
t_target = 0.5

# ============================================================
# 7. CREATE FIGURE
# ============================================================
#fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

# ============================================================
# 8. EVALUATION LOOP
# ============================================================
for asset in range(num_assets):

    ax = axes[asset]
    asset_id = asset + 1
    other_id = 2 if asset_id == 1 else 1

    for q_fixed in fixed_other_values:

        delta_minus_curve = []
        delta_plus_curve = []

        for q in q_values:

            if asset_id == 1:
                q1, q2 = q, q_fixed
            else:
                q1, q2 = q_fixed, q

            obs = build_obs(q1, q2, t_target)
            action, _ = model.predict(obs, deterministic=True)
            action = np.asarray(action).reshape(1, num_assets, 2)

            delta_minus_curve.append(action[0, asset, 0])
            delta_plus_curve.append(action[0, asset, 1])

        delta_minus_curve = np.array(delta_minus_curve)
        delta_plus_curve  = np.array(delta_plus_curve)

        ax.plot(
            q_values,
            delta_minus_curve,
            linestyle="-",
            label=rf"$\delta_{{{asset_id}}}^-$ | $q_{{{other_id}}}={q_fixed}$"
        )

        ax.plot(
            q_values,
            delta_plus_curve,
            linestyle="--",
            label=rf"$\delta_{{{asset_id}}}^+$ | $q_{{{other_id}}}={q_fixed}$"
        )

    # ----------------------------
    # Titles REMOVED (commented)
    # ----------------------------
    # ax.set_title(
    #     rf"$\delta_{{{asset_id}}}^-$ and "
    #     rf"$\delta_{{{asset_id}}}^+$ vs "
    #     rf"$q_{{{asset_id}}}$"
    #     rf"  ($t={t_target}$)"
    # )

    ax.set_xlabel(rf"$q_{{{asset_id}}}$")
    ax.set_ylabel(
        rf"$\delta_{{{asset_id}}}^-$ and "
        rf"$\delta_{{{asset_id}}}^+$"
    )

    ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
    ax.tick_params(axis='both', which='major', width=1.5, length=6)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('medium')

    ax.legend(ncol=2, frameon=True)

# ============================================================
# 9. SUPER TITLE REMOVED (commented)
# ============================================================
# fig.suptitle(
#     rf"{run_tag}",
#     fontsize=16
# )

# ============================================================
# 10. SAVE IMAGES
# ============================================================
t_str = int(t_target * 100)
base_name = f"{run_tag}_LineSlices_t{t_str}_Combined"

pdf_path = os.path.join(fig_dir, base_name + ".pdf")
png_path = os.path.join(fig_dir, base_name + ".png")

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=600, bbox_inches="tight")

# ============================================================
# 11. SAVE METADATA (NEW INTEGRATION)
# ============================================================
meta = {
    "run_tag": run_tag,
    "trial_folder": trial_folder,
    "assets_tag": assets_tag,
    "source_model_path": model_path,
    "t_target": float(t_target),
    "fixed_other_values": fixed_other_values,
    "fixed_state_values": fixed_state_values.tolist(),
    "q_min": int(q_values.min()),
    "q_max": int(q_values.max()),
    "files": {
        "figure_pdf": os.path.basename(pdf_path),
        "figure_png": os.path.basename(png_path),
    }
}

meta_json_path = os.path.join(data_dir, base_name + "_meta.json")
with open(meta_json_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("\nSaved Publication Figure and Metadata:")
print("  PDF :", pdf_path)
print("  PNG :", png_path)
print("  Meta JSON:", meta_json_path)

plt.show()
plt.close(fig)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# N=2 case 
# NOTE: not used in the (first version of the Manuscript submitted to Fututes Market) but can be useful
# A) "Spread & Center" decomposition plots (multi-asset)
# ONE FIGURE PER ASSET:
#   Left:  Half-Spread s_i vs inventory q_i at fixed time t_target
#   Right: Skew/Center k_i vs inventory q_i at fixed time t_target
#
# Uses saved contour arrays:
#   delta_minus(q,t), delta_plus(q,t)
#
# Definitions:
#   s_i(q,t) = 0.5 * (delta_plus_i(q,t) + delta_minus_i(q,t))
#   k_i(q,t) = 0.5 * (delta_plus_i(q,t) - delta_minus_i(q,t))
#
# Loads from:
#   data_root\trial_folder\{run_tag}_Asset{j}_delta_minus.txt
#   data_root\trial_folder\{run_tag}_Asset{j}_delta_plus.txt
#
# Saves to:
#   out_root\trial_folder\{run_tag}_Asset{j}_spread_center_t{t_used}.pdf/png
# ============================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# PATHS (edit these)
# ----------------------------
data_root = os.path.join(PROJECT_ROOT, "N_figures/2Assets/contour_data")
out_root  = os.path.join(PROJECT_ROOT, "N_figures/2Assets/spread_center")

run_tag = "PPO_2Assets_Trial1_100M"  # must match saved contour files
num_assets = 2

# ----------------------------
# Infer trial folder from run_tag
# Examples:
#   PPO_2Assets_Trial3_65M        -> PPO_2Assets_Trial3
#   PPO_2Assets_Trial3_Best_Model -> PPO_2Assets_Trial3
# ----------------------------
def infer_trial_folder_from_run_tag(tag: str) -> str:
    m = re.match(r"^(PPO_\d+Assets_Trial\d+)", tag)
    if m:
        return m.group(1)
    parts = tag.split("_")
    return "_".join(parts[:-1]) if len(parts) >= 2 else tag

trial_folder = infer_trial_folder_from_run_tag(run_tag)

# Final dirs
data_dir = os.path.join(data_root, trial_folder)
out_dir  = os.path.join(out_root, trial_folder)
os.makedirs(out_dir, exist_ok=True)

print("Using:")
print("  trial_folder:", trial_folder)
print("  data_dir     :", data_dir)
print("  out_dir      :", out_dir)

# ----------------------------
# GRID (must match contour script)
# ----------------------------
q_values = np.arange(-10, 11, 1)     # -10..10 inclusive
t_values = np.linspace(0, 1, 101)    # 101 points

# ----------------------------
# Pick t_target (closest grid point)
# ----------------------------
t_target = 0.5
t_index = int(np.argmin(np.abs(t_values - t_target)))
t_used = float(t_values[t_index])
print(f"Using t_index={t_index}, t_value={t_used}")

# ----------------------------
# Optional smoothing for nicer lines
# ----------------------------
def smooth_1d(y: np.ndarray, window: int = 1) -> np.ndarray:
    """
    Simple moving-average smoothing.
    window must be odd and >= 3 to smooth.
    window=1 => no smoothing
    """
    if window is None or window <= 1:
        return y
    window = int(window)
    if window % 2 == 0:
        window += 1
    if window >= len(y):
        window = max(3, len(y) // 2 * 2 + 1)

    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(ypad, kernel, mode="valid")

smooth_window = 1  # try 5 or 7 if you want smoothing

# ----------------------------
# Loop over assets
# ----------------------------
for asset in range(num_assets):
    asset_tag = f"Asset{asset+1}"

    # Load saved delta arrays
    dm_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_minus.txt")
    dp_path = os.path.join(data_dir, f"{run_tag}_{asset_tag}_delta_plus.txt")

    if not os.path.exists(dm_path):
        raise FileNotFoundError(f"Missing file: {dm_path}")
    if not os.path.exists(dp_path):
        raise FileNotFoundError(f"Missing file: {dp_path}")

    delta_minus = np.loadtxt(dm_path, delimiter=",")
    delta_plus  = np.loadtxt(dp_path, delimiter=",")

    expected_shape = (len(q_values), len(t_values))
    if delta_minus.shape != expected_shape:
        raise ValueError(f"{asset_tag} delta_minus shape {delta_minus.shape} != expected {expected_shape}")
    if delta_plus.shape != expected_shape:
        raise ValueError(f"{asset_tag} delta_plus shape {delta_plus.shape} != expected {expected_shape}")

    # Extract curves at chosen time
    dm_t = delta_minus[:, t_index]  # shape (len(q),)
    dp_t = delta_plus[:,  t_index]  # shape (len(q),)

    # Spread/Center decomposition
    half_spread = 0.5 * (dp_t + dm_t)  # s_i(q_i, t)
    skew_center = 0.5 * (dp_t - dm_t)  # k_i(q_i, t)

    # Optional smoothing
    half_spread_s = smooth_1d(half_spread, window=smooth_window)
    skew_center_s = smooth_1d(skew_center, window=smooth_window)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    # Left: half-spread
    axes[0].plot(q_values, half_spread_s)
    axes[0].set_title(
        rf"Asset {asset+1}: Half-Spread vs Inventory $q_{{{asset+1}}}$ at $t \approx {t_used:.2f}$",
        fontsize=12
    )
    axes[0].set_xlabel(rf"Inventory $q_{{{asset+1}}}$", fontsize=11)
    axes[0].set_ylabel(rf"Half-Spread $s_{{{asset+1}}}$", fontsize=11)
    axes[0].grid(True)
    axes[0].tick_params(labelsize=10)

    # Right: skew/center
    axes[1].plot(q_values, skew_center_s)
    axes[1].set_title(
        rf"Asset {asset+1}: Skew/Center vs Inventory $q_{{{asset+1}}}$ at $t \approx {t_used:.2f}$",
        fontsize=12
    )
    axes[1].set_xlabel(rf"Inventory $q_{{{asset+1}}}$", fontsize=11)
    axes[1].set_ylabel(rf"Skew/Center $k_{{{asset+1}}}$", fontsize=11)
    axes[1].grid(True)
    axes[1].tick_params(labelsize=10)

    fig.suptitle(
        f"Spread–Center decomposition — {run_tag} — {asset_tag}",
        fontsize=13
    )

    # Save
    pdf_path = os.path.join(out_dir, f"{run_tag}_{asset_tag}_spread_center_t{t_used:.2f}.pdf")
    png_path = os.path.join(out_dir, f"{run_tag}_{asset_tag}_spread_center_t{t_used:.2f}.png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"\nSaved spread-center plots for {asset_tag}:")
    print("  PDF:", pdf_path)
    print("  PNG:", png_path)

    plt.show()

Single Asset (N=1) Case

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================
# N=1 case : Contor Plot same as N=2 case
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# Contour plots (side-by-side) + separate colorbars
# SAME colormap + SAME vmin/vmax + save data + save a readable META TXT
# SINGLE-ASSET (N=1) using the MULTI-ASSET codebase checkpoints
# Saves under ...\SingleAsset\...\<trial_folder>\  NEW
# Works for BOTH: checkpoint_path OR best_model_path
# If best_model_path is loaded -> run_tag becomes: <trial_folder>_Best_Model
# ============================

import os
import re
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

# ----------------------------
# Paths
# ----------------------------
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_SingleAsset/PPO_1Assets_Trial22/PPO_1Assets_Trial22_30M.zip")
best_model_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Best_SingleAsset/PPO_1Assets_Trial1/best_model.zip")

# ----------------------------
# Choose what you load (set ONE of these)
# ----------------------------
use_best_model = False  # True => load best_model.zip, False => load checkpoint

model_path = best_model_path if use_best_model else checkpoint_path
model = PPO.load(model_path)

# ----------------------------
# Infer run_tag
# - checkpoint: use filename without extension (e.g. PPO_1Assets_Trial1_30M)
# - best_model: use trial folder name + "_Best_Model" (e.g. PPO_1Assets_Trial1_Best_Model)
# ----------------------------
def infer_run_tag_for_best(best_path: str) -> str:
    trial_folder = os.path.basename(os.path.dirname(best_path))  # PPO_1Assets_Trial1
    return f"{trial_folder}_Best_Model"

def infer_run_tag_for_checkpoint(ckpt_path: str) -> str:
    return os.path.splitext(os.path.basename(ckpt_path))[0]

run_tag = infer_run_tag_for_best(best_model_path) if use_best_model else infer_run_tag_for_checkpoint(checkpoint_path)

# ----------------------------
# NEW: Infer trial_folder from whichever path you loaded
# We prefer the folder that contains the checkpoint/best_model.zip:
#   checkpoint: ...\PPO_Checkpoints_SingleAsset\<trial_folder>\<file>.zip
#   best:       ...\PPO_Best_SingleAsset\<trial_folder>\best_model.zip
# ----------------------------
trial_folder = os.path.basename(os.path.dirname(model_path))

# Safety fallback: parse from run_tag if needed
def infer_trial_folder_from_run_tag(tag: str) -> str:
    m = re.match(r"^(PPO_\d+Assets_Trial\d+)", tag)
    if m:
        return m.group(1)
    parts = tag.split("_")
    return "_".join(parts[:-1]) if len(parts) >= 2 else tag

if not trial_folder.startswith("PPO_") or "Trial" not in trial_folder:
    trial_folder = infer_trial_folder_from_run_tag(run_tag)

print("Using:")
print("  run_tag     :", run_tag)
print("  trial_folder:", trial_folder)

# ----------------------------
# NEW: Save dirs with trial subfolder
# ----------------------------
fig_root  = os.path.join(PROJECT_ROOT, "N_figures/SingleAsset/contour")
data_root = os.path.join(PROJECT_ROOT, "N_figures/SingleAsset/contour_data")

fig_dir  = os.path.join(fig_root, trial_folder)
data_dir = os.path.join(data_root, trial_folder)

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print("  fig_dir     :", fig_dir)
print("  data_dir    :", data_dir)

# ----------------------------
# Fixed values + grid
# observation (extended reduced state) = [q, t, lambda_bid, lambda_ask, c_bid, c_ask]
# ----------------------------
num_assets = 1
fixed_state_values = np.array([18.2, 18.2, 3.72, 3.72], dtype=float)  # [λ_bid, λ_ask, c_bid, c_ask]

q_values = np.arange(-10, 11, 1)        # -10..10 inclusive
t_values = np.linspace(0, 1, 101)       # 101 points => 100 intervals

delta_minus = np.zeros((len(q_values), len(t_values)))
delta_plus  = np.zeros((len(q_values), len(t_values)))

# Detect what obs dim this checkpoint expects
obs_dim = None
if getattr(model, "observation_space", None) is not None:
    obs_dim = model.observation_space.shape[0]

# ----------------------------
# Main evaluation loop
# ----------------------------
for i, q in enumerate(q_values):
    for j, t in enumerate(t_values):
        # Case A: minimal reduced state [q, t]
        if obs_dim == 2:
            obs = np.array([[q, t]], dtype=float)

        # Case B: extended reduced state [q, t, λ_bid, λ_ask, c_bid, c_ask]
        elif obs_dim == 6:
            obs = np.concatenate([[q], [t], fixed_state_values]).reshape(1, -1)

        # Fallback: assume extended style
        else:
            obs = np.concatenate([[q], [t], fixed_state_values]).reshape(1, -1)

        action, _ = model.predict(obs, deterministic=True)

        # Force shape to (1, num_assets, 2)
        action = np.asarray(action).reshape(1, num_assets, 2)

        delta_minus[i, j] = action[0, 0, 0]
        delta_plus[i, j]  = action[0, 0, 1]

print("obs_dim:", obs_dim)
print("q range:", q_values.min(), q_values.max(), "len:", len(q_values))
print("t range:", t_values.min(), t_values.max(), "len:", len(t_values))
print("fixed_state_values:", fixed_state_values)
print("delta_minus min/max:", float(delta_minus.min()), float(delta_minus.max()))
print("delta_plus  min/max:", float(delta_plus.min()),  float(delta_plus.max()))

# ----------------------------
# Plot settings: SAME colormap + SAME vmin/vmax, but TWO separate colorbars
# ----------------------------
vmin = float(min(delta_minus.min(), delta_plus.min()))
vmax = float(max(delta_minus.max(), delta_plus.max()))
levels = np.linspace(vmin, vmax, 60)
cmap_shared = "viridis"

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

# Left: delta_minus
cf1 = axes[0].contourf(
    t_values, q_values, delta_minus,
    levels=levels, cmap=cmap_shared, vmin=vmin, vmax=vmax
)
axes[0].set_title(r"$\delta^{-}$ (Buy order distance)", fontsize=12)
axes[0].set_xlabel("Time (t)", fontsize=11)
axes[0].set_ylabel("Inventory (q)", fontsize=11)
axes[0].tick_params(labelsize=10)
cbar1 = fig.colorbar(cf1, ax=axes[0], pad=0.02)
cbar1.set_label(r"$\delta^{-}$", fontsize=11)
cbar1.ax.tick_params(labelsize=10)

# Right: delta_plus
cf2 = axes[1].contourf(
    t_values, q_values, delta_plus,
    levels=levels, cmap=cmap_shared, vmin=vmin, vmax=vmax
)
axes[1].set_title(r"$\delta^{+}$ (Sell order distance)", fontsize=12)
axes[1].set_xlabel("Time (t)", fontsize=11)
axes[1].set_ylabel("Inventory (q)", fontsize=11)
axes[1].tick_params(labelsize=10)
cbar2 = fig.colorbar(cf2, ax=axes[1], pad=0.02)
cbar2.set_label(r"$\delta^{+}$", fontsize=11)
cbar2.ax.tick_params(labelsize=10)

fig.suptitle(f"Policy contours — {run_tag}", fontsize=13)

# ----------------------------
# SAVE ALWAYS (both checkpoint or best)
# ----------------------------
# Figures
fig_path_pdf = os.path.join(fig_dir,  f"{run_tag}.pdf")  # vector (best for papers)
fig_path_png = os.path.join(fig_dir,  f"{run_tag}.png")  # high-res raster
fig.savefig(fig_path_pdf, bbox_inches="tight")
fig.savefig(fig_path_png, dpi=600, bbox_inches="tight")

# Data
dm_path = os.path.join(data_dir, f"{run_tag}_delta_minus.txt")
dp_path = os.path.join(data_dir, f"{run_tag}_delta_plus.txt")
np.savetxt(dm_path, delta_minus, delimiter=",")
np.savetxt(dp_path, delta_plus, delimiter=",")

# Metadata (easy to open in Notepad)
meta_path = os.path.join(data_dir, f"{run_tag}_meta.txt")
with open(meta_path, "w", encoding="utf-8") as f:
    f.write(f"run_tag: {run_tag}\n")
    f.write(f"trial_folder: {trial_folder}\n")
    f.write(f"use_best_model: {use_best_model}\n")
    f.write(f"source_model_path: {model_path}\n")
    f.write(f"obs_dim_detected: {obs_dim}\n")
    f.write(f"delta_minus_file: {os.path.basename(dm_path)}\n")
    f.write(f"delta_plus_file : {os.path.basename(dp_path)}\n")
    f.write("\n")
    f.write("GRID\n")
    f.write(f"q_min: {int(q_values.min())}\n")
    f.write(f"q_max: {int(q_values.max())}\n")
    f.write(f"q_step: {int(q_values[1] - q_values[0])}\n")
    f.write(f"q_num_points: {len(q_values)}\n")
    f.write("\n")
    f.write(f"t_min: {float(t_values.min())}\n")
    f.write(f"t_max: {float(t_values.max())}\n")
    f.write(f"t_num_points: {len(t_values)}\n")
    f.write(f"t_num_intervals: {len(t_values) - 1}\n")
    f.write("\n")
    f.write("FIXED STATE VALUES\n")
    f.write("If obs_dim == 2, only [q, t] was used.\n")
    f.write("If obs_dim == 6, observation = [q, t, lambda_bid, lambda_ask, c_bid, c_ask]\n")
    f.write(f"lambda_bid: {fixed_state_values[0]}\n")
    f.write(f"lambda_ask: {fixed_state_values[1]}\n")
    f.write(f"c_bid     : {fixed_state_values[2]}\n")
    f.write(f"c_ask     : {fixed_state_values[3]}\n")
    f.write("\n")
    f.write("PLOT SETTINGS\n")
    f.write(f"colormap: {cmap_shared}\n")
    f.write(f"vmin: {vmin}\n")
    f.write(f"vmax: {vmax}\n")
    f.write(f"num_levels: {len(levels)}\n")

print("Saved:")
print("  Figure PDF:", fig_path_pdf)
print("  Figure PNG:", fig_path_png)
print("  Data delta_minus:", dm_path)
print("  Data delta_plus :", dp_path)
print("  Meta TXT:", meta_path)

plt.show()


In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# OVER TIME(N=1) : Same as N=2 case
# ONE FIGURE: δ⁻ and δ⁺ over time (side-by-side) for several q values (N=1)
# Standalone: loads saved arrays + rebuilds q_values/t_values + smooth lines
# UPDATED: uses trial folder layer in BOTH data_dir and out_dir
# Expects:
#   ...\SingleAsset\contour_data\<trial_folder>\{run_tag}_delta_minus.txt
# Saves to:
#   ...\SingleAsset\over_time\<trial_folder>\{run_tag}_over_time_smooth.(pdf|png)
# ============================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# PATH ROOTS
# ----------------------------
data_root = os.path.join(PROJECT_ROOT, "N_figures/SingleAsset/contour_data")
out_root  = os.path.join(PROJECT_ROOT, "N_figures/SingleAsset/over_time")

run_tag = "PPO_1Assets_Trial22_30M"  # <-- must match the saved files (and imply trial)

# ----------------------------
# Infer trial folder from run_tag
# Example: "PPO_1Assets_Trial8_30M" -> "PPO_1Assets_Trial8"
# If run_tag ends with "_Best_Model", it will still infer "PPO_1Assets_Trial8".
# ----------------------------
def infer_trial_folder_from_run_tag(tag: str) -> str:
    m = re.match(r"^(PPO_\d+Assets_Trial\d+)", tag)
    if m:
        return m.group(1)
    # fallback: best-effort
    parts = tag.split("_")
    if len(parts) >= 3:
        return "_".join(parts[:3])  # PPO, 1Assets, Trial8
    return tag

trial_folder = infer_trial_folder_from_run_tag(run_tag)

# ----------------------------
# Final dirs with trial subfolder
# ----------------------------
data_dir = os.path.join(data_root, trial_folder)
out_dir  = os.path.join(out_root,  trial_folder)
os.makedirs(out_dir, exist_ok=True)

print("Using:")
print("  run_tag     :", run_tag)
print("  trial_folder:", trial_folder)
print("  data_dir    :", data_dir)
print("  out_dir     :", out_dir)

# ----------------------------
# GRID (must match contour script)
# ----------------------------
q_values = np.arange(-10, 11, 1)     # -10..10 inclusive
t_values = np.linspace(0, 1, 101)    # 101 points

# ----------------------------
# Load saved arrays
# ----------------------------
dm_path = os.path.join(data_dir, f"{run_tag}_delta_minus.txt")
dp_path = os.path.join(data_dir, f"{run_tag}_delta_plus.txt")

if not os.path.exists(dm_path):
    raise FileNotFoundError(f"Missing file: {dm_path}")
if not os.path.exists(dp_path):
    raise FileNotFoundError(f"Missing file: {dp_path}")

delta_minus = np.loadtxt(dm_path, delimiter=",")
delta_plus  = np.loadtxt(dp_path, delimiter=",")

# ----------------------------
# Sanity checks
# ----------------------------
expected_shape = (len(q_values), len(t_values))
if delta_minus.shape != expected_shape:
    raise ValueError(
        f"delta_minus shape {delta_minus.shape} != expected {expected_shape}. "
        f"Check q_values/t_values here vs how contour data was saved."
    )
if delta_plus.shape != expected_shape:
    raise ValueError(
        f"delta_plus shape {delta_plus.shape} != expected {expected_shape}. "
        f"Check q_values/t_values here vs how contour data was saved."
    )

# ----------------------------
# Choose q slices to visualize
# ----------------------------
q_subset = np.arange(-3, 4, 1)   # e.g. -3..3

# ----------------------------
# (Optional) smoothing helper (moving average)
# ----------------------------
def smooth_1d(y: np.ndarray, window: int = 7) -> np.ndarray:
    """
    Simple moving-average smoothing.
    window must be odd and >= 3 to look nice.
    """
    if window is None or window <= 1:
        return y
    window = int(window)
    if window % 2 == 0:
        window += 1
    if window >= len(y):
        window = max(3, len(y) // 2 * 2 + 1)  # largest odd < len(y)

    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(ypad, kernel, mode="valid")

smooth_window = 1  # 1 = no smoothing; try 5,7,9

# -----------------------------------------------------------
# Plot side-by-side (one figure)
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

# δ⁻ panel
for q in q_subset:
    idx = np.where(q_values == q)[0]
    if len(idx) == 0:
        continue
    q_index = idx[0]
    y = delta_minus[q_index, :]
    y_s = smooth_1d(y, window=smooth_window)
    axes[0].plot(t_values, y_s, label=f"q={q}")

axes[0].set_title(r"$\delta^{-}$ over time (Buy order distance)", fontsize=12)
axes[0].set_xlabel("Time (t)", fontsize=11)
axes[0].set_ylabel(r"$\delta^{-}$", fontsize=11)
axes[0].grid(True)
axes[0].tick_params(labelsize=10)
axes[0].legend(fontsize=9, ncol=2, frameon=True)

# δ⁺ panel
for q in q_subset:
    idx = np.where(q_values == q)[0]
    if len(idx) == 0:
        continue
    q_index = idx[0]
    y = delta_plus[q_index, :]
    y_s = smooth_1d(y, window=smooth_window)
    axes[1].plot(t_values, y_s, label=f"q={q}")

axes[1].set_title(r"$\delta^{+}$ over time (Sell order distance)", fontsize=12)
axes[1].set_xlabel("Time (t)", fontsize=11)
axes[1].set_ylabel(r"$\delta^{+}$", fontsize=11)
axes[1].grid(True)
axes[1].tick_params(labelsize=10)
axes[1].legend(fontsize=9, ncol=2, frameon=True)

fig.suptitle(
    f"Policy over-time curves — {run_tag} (q = {q_subset.min()}..{q_subset.max()})",
    fontsize=13
)

# ----------------------------
# Save (publication quality) into trial folder
# ----------------------------
pdf_path = os.path.join(out_dir, f"{run_tag}_over_time_smooth.pdf")
png_path = os.path.join(out_dir, f"{run_tag}_over_time_smooth.png")

# Optional: same y-limits for both panels
ymin = float(min(delta_minus.min(), delta_plus.min()))
ymax = float(max(delta_minus.max(), delta_plus.max()))
axes[0].set_ylim(ymin, ymax)
axes[1].set_ylim(ymin, ymax)

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=600, bbox_inches="tight")

print("Saved combined over-time plots:")
print("  PDF:", pdf_path)
print("  PNG:", png_path)

plt.show()


In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# N=1 :  MID TIME : Same as N=2 case
# STANDALONE: PLOT δ⁻ AND δ⁺ OVER q AT t = 0.5 (N = 1 asset)
# - loads saved delta arrays from contour_data
# - rebuilds q_values/t_values grid
# - saves PDF + high-res PNG under ...\SingleAsset\mid_time
# UPDATED: uses trial folder layer in BOTH data_dir and out_dir
# Expects:
#   ...\SingleAsset\contour_data\<trial_folder>\{run_tag}_delta_minus.txt
# Saves to:
#   ...\SingleAsset\mid_time\<trial_folder>\{run_tag}_mid_time_tXX.(pdf|png)
# ============================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# PATH ROOTS
# ----------------------------
data_root = os.path.join(PROJECT_ROOT, "N_figures/SingleAsset/contour_data")
out_root  = os.path.join(PROJECT_ROOT, "N_figures/SingleAsset/mid_time")

run_tag = "PPO_1Assets_Trial19_100M"  # <-- must match the saved files (and imply trial)

# ----------------------------
# Infer trial folder from run_tag
# Example: "PPO_1Assets_Trial1_30M" -> "PPO_1Assets_Trial1"
# If run_tag ends with "_Best_Model", it will still infer "PPO_1Assets_Trial1".
# ----------------------------
def infer_trial_folder_from_run_tag(tag: str) -> str:
    m = re.match(r"^(PPO_\d+Assets_Trial\d+)", tag)
    if m:
        return m.group(1)
    parts = tag.split("_")
    if len(parts) >= 3:
        return "_".join(parts[:3])
    return tag

trial_folder = infer_trial_folder_from_run_tag(run_tag)

# ----------------------------
# Final dirs with trial subfolder
# ----------------------------
data_dir = os.path.join(data_root, trial_folder)
out_dir  = os.path.join(out_root,  trial_folder)
os.makedirs(out_dir, exist_ok=True)

print("Using:")
print("  run_tag     :", run_tag)
print("  trial_folder:", trial_folder)
print("  data_dir    :", data_dir)
print("  out_dir     :", out_dir)

# ----------------------------
# GRID (must match contour script)
# ----------------------------
q_values = np.arange(-10, 11, 1)     # -10..10 inclusive
t_values = np.linspace(0, 1, 101)    # 101 points

# ----------------------------
# Load saved arrays
# ----------------------------
dm_path = os.path.join(data_dir, f"{run_tag}_delta_minus.txt")
dp_path = os.path.join(data_dir, f"{run_tag}_delta_plus.txt")

if not os.path.exists(dm_path):
    raise FileNotFoundError(f"Missing file: {dm_path}")
if not os.path.exists(dp_path):
    raise FileNotFoundError(f"Missing file: {dp_path}")

delta_minus = np.loadtxt(dm_path, delimiter=",")
delta_plus  = np.loadtxt(dp_path, delimiter=",")

# ----------------------------
# Sanity checks
# ----------------------------
expected_shape = (len(q_values), len(t_values))
if delta_minus.shape != expected_shape:
    raise ValueError(
        f"delta_minus shape {delta_minus.shape} != expected {expected_shape}. "
        f"Check q_values/t_values here vs how contour data was saved."
    )
if delta_plus.shape != expected_shape:
    raise ValueError(
        f"delta_plus shape {delta_plus.shape} != expected {expected_shape}. "
        f"Check q_values/t_values here vs how contour data was saved."
    )

# ----------------------------
# Pick t = 0.5 (closest grid point)
# ----------------------------
t_target = 0.5
t_index = int(np.argmin(np.abs(t_values - t_target)))
t_used = float(t_values[t_index])
print(f"Using t_index = {t_index}, t_value = {t_used}")

delta_minus_t = delta_minus[:, t_index]   # (len(q),)
delta_plus_t  = delta_plus[:,  t_index]   # (len(q),)

# ----------------------------
# Plot (smooth line style, no markers)
# ----------------------------
plt.figure(figsize=(12, 6))
plt.plot(q_values, delta_minus_t, label=rf"$\delta^-(t={t_used:.2f})$")
plt.plot(q_values, delta_plus_t,  label=rf"$\delta^+(t={t_used:.2f})$")

plt.title(
    rf"Single Asset — $\delta^-$ and $\delta^+$ vs Inventory at $t \approx {t_used:.2f}$",
    fontsize=12
)
plt.xlabel("Inventory q", fontsize=11)
plt.ylabel("Quote Distance", fontsize=11)
plt.grid(True)
plt.legend()
plt.tight_layout()

# ----------------------------
# Save (publication quality) into trial folder
# ----------------------------
pdf_path = os.path.join(out_dir, f"{run_tag}_mid_time_t{t_used:.2f}.pdf")
png_path = os.path.join(out_dir, f"{run_tag}_mid_time_t{t_used:.2f}.png")

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=600, bbox_inches="tight")

print("Saved mid-time plots:")
print("  PDF:", pdf_path)
print("  PNG:", png_path)

plt.show()


N=3 Case 

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# N=3 case. Contour plot 
# ============================
# Very Important : Fixed State Vales in the code below are different depending on checkpoint (Model Trail Number)
# Very Important : Most Check Points Have Same Fixed State Values but not all of them
# Very Important : Fixed State Vales are computed in "N_Validation_Comprehensive" NoteBook. 
# Very Important : If you train a new model with new paramters, recompute Fixed State Vales before plotting
# Very Important:  Fixed State Vales are basicallly average values (average over time and across simulations) 


# ============================
# Contour plots (side-by-side) + separate colorbars
# SAME colormap + SAME vmin/vmax
# MULTI-ASSET (N=3) using the MULTI-ASSET codebase checkpoints
# NO SAVING (just plot)
# Produces ONE figure per asset: [delta_minus | delta_plus]
# ============================

import os
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

# ----------------------------
# Paths (edit these)
# ----------------------------
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_MultiAsset/PPO_3Assets_Trial1/PPO_3Assets_Trial1_100M.zip")
best_model_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Best_MultiAsset/PPO_3Assets_TrialX/best_model.zip")

# ----------------------------
# Choose what you load (set ONE)
# ----------------------------
use_best_model = False  # True => load best_model.zip, False => load checkpoint

model_path = best_model_path if use_best_model else checkpoint_path
model = PPO.load(model_path)

# ----------------------------
# Infer run_tag
# ----------------------------
def infer_run_tag_for_best(best_path: str) -> str:
    trial_folder = os.path.basename(os.path.dirname(best_path))  # PPO_3Assets_TrialX
    return f"{trial_folder}_Best_Model"

def infer_run_tag_for_checkpoint(ckpt_path: str) -> str:
    return os.path.splitext(os.path.basename(ckpt_path))[0]

run_tag = infer_run_tag_for_best(best_model_path) if use_best_model else infer_run_tag_for_checkpoint(checkpoint_path)

# ----------------------------
# Fixed values + grid
# Reduced state (extended) expected for N=3:
# obs = [q1, q2, q3, t,
#        λ1_bid, λ1_ask, λ2_bid, λ2_ask, λ3_bid, λ3_ask,
#        c1_bid, c1_ask, c2_bid, c2_ask, c3_bid, c3_ask]
# ----------------------------
num_assets = 3

# Format: [λ1_bid, λ1_ask, λ2_bid, λ2_ask, λ3_bid, λ3_ask, c1_bid, c1_ask, c2_bid, c2_ask, c3_bid, c3_ask]
fixed_state_values = np.array([
    # --- MO intensities (post-burn means you gave) ---
    15.3474, 15.3492,   # Asset 0: bid, ask
    17.5669, 17.5663,   # Asset 1: bid, ask
    19.8150, 19.8132,   # Asset 2: bid, ask

    # --- LOB depths (post-burn means you gave) ---
    3.6542,  3.6547,    # Asset 0: bid, ask
    4.2104,  4.2101,    # Asset 1: bid, ask
    4.7790,  4.7785,    # Asset 2: bid, ask
], dtype=float)

q_values = np.arange(-10, 11, 1)    # -10..10 inclusive
t_values = np.linspace(0, 1, 101)   # 101 points => 100 intervals

# Detect what obs dim this checkpoint expects
obs_dim = None
if getattr(model, "observation_space", None) is not None:
    obs_dim = model.observation_space.shape[0]
print("Detected obs_dim:", obs_dim)

# ----------------------------
# Helper: build obs for N=3
# ----------------------------
def build_obs_for_asset(asset_index: int, q: float, t: float) -> np.ndarray:
    """
    asset_index: 0,1,2
    Vary only q_asset_index = q; freeze the other inventories at 0.
    """
    q_vec = np.zeros(num_assets, dtype=float)
    q_vec[asset_index] = float(q)

    # Case A: minimal reduced state (if trained on [q1,q2,q3,t])
    if obs_dim == (num_assets + 1):  # 4
        obs = np.concatenate([q_vec, [t]]).reshape(1, -1)

    # Case B: extended reduced state [q(3), t, lam(6), dep(6)] => 3 + 1 + 6 + 6 = 16
    elif obs_dim == (5 * num_assets + 1):  # 16
        obs = np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)

    # Fallback: assume extended
    else:
        obs = np.concatenate([q_vec, [t], fixed_state_values]).reshape(1, -1)

    return obs

# ----------------------------
# Plot settings
# ----------------------------
cmap_shared = "viridis"

# ----------------------------
# Main evaluation + plotting per asset
# ----------------------------
for asset in range(num_assets):

    delta_minus = np.zeros((len(q_values), len(t_values)))
    delta_plus  = np.zeros((len(q_values), len(t_values)))

    # --- evaluate policy over grid ---
    for i, q in enumerate(q_values):
        for j, t in enumerate(t_values):
            obs = build_obs_for_asset(asset, q, t)

            action, _ = model.predict(obs, deterministic=True)
            action = np.asarray(action).reshape(1, num_assets, 2)

            delta_minus[i, j] = action[0, asset, 0]
            delta_plus[i, j]  = action[0, asset, 1]

    # diagnostics
    print(f"\nAsset {asset}:")
    print("  delta_minus min/max:", float(delta_minus.min()), float(delta_minus.max()))
    print("  delta_plus  min/max:", float(delta_plus.min()),  float(delta_plus.max()))

    # shared scaling across δ⁻ and δ⁺ for THIS asset
    vmin = float(min(delta_minus.min(), delta_plus.min()))
    vmax = float(max(delta_minus.max(), delta_plus.max()))
    levels = np.linspace(vmin, vmax, 60)

    # --- side-by-side figure ---
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    # Left: delta_minus
    cf1 = axes[0].contourf(
        t_values, q_values, delta_minus,
        levels=levels, cmap=cmap_shared, vmin=vmin, vmax=vmax
    )
    axes[0].set_title(rf"Asset {asset}: $\delta^{{-}}$ (Buy order distance)", fontsize=12)
    axes[0].set_xlabel("Time (t)", fontsize=11)
    axes[0].set_ylabel("Inventory (q)", fontsize=11)
    cbar1 = fig.colorbar(cf1, ax=axes[0], pad=0.02)
    cbar1.set_label(r"$\delta^{-}$", fontsize=11)

    # Right: delta_plus
    cf2 = axes[1].contourf(
        t_values, q_values, delta_plus,
        levels=levels, cmap=cmap_shared, vmin=vmin, vmax=vmax
    )
    axes[1].set_title(rf"Asset {asset}: $\delta^{{+}}$ (Sell order distance)", fontsize=12)
    axes[1].set_xlabel("Time (t)", fontsize=11)
    axes[1].set_ylabel("Inventory (q)", fontsize=11)
    cbar2 = fig.colorbar(cf2, ax=axes[1], pad=0.02)
    cbar2.set_label(r"$\delta^{+}$", fontsize=11)

    fig.suptitle(f"Policy contours — {run_tag} — Asset {asset}", fontsize=13)

    plt.show()
